#### 0. CONFIGURAR AMBIENTE

In [56]:
# AMBIENTE NOTEBOOK

## Remover avisos de carregamento de pacotes
suppressPackageStartupMessages({
  library(tidyverse)
  library(summarytools)
  library(kableExtra)
  library(stargazer)
  library(plm)
  library(fixest)
})

## Configurar exibição no Jupyter Notebook
options(
  repr.matrix.max.cols = Inf,
  tibble.print_max = Inf,
  data.frame.print.max = Inf,
  max.print = 999999,
  jupyter.display_mimetypes = setdiff(
    getOption("jupyter.display_mimetypes"),
    "text/latex"
  )
)

In [ ]:
# PACOTES

## Dfs, tabelas e gráficos
library(tidyverse)
library(summarytools) # Resumos descritivos de dados
library(kableExtra) # Tabelas HTML/LaTeX bonitas
library(stargazer) # Gerar tabelas resumo
library(hrbrthemes) # Temas gráficos para ggplot2
library(ggcorrplot) # Gráficos de correlação
library(DT) # Tabelas interativas
library(knitr) # Gerar tabelas em HTML/LaTeX

## Efeitos fixos
library(plm) # simples ou two-way (existem duas dimensões de efeitos fixos)
library(fixest) # within (transformação utilizada na estimação) (MELHOR)

# Sintaxe do fixest: y ~ x1 + x2 | fe1 + fe2, data = df, vcov = ~cluster_var
# https://lrberge.github.io/fixest/reference/feols.html

***

#### 1. DATAFRAME PRINCIPAL

In [58]:
### Dcionário prefixos
## 0: IDENTIFICADORES

## 1. AMBIENTAIS
# 1a: Conversão
# 1b: Estoque
# 1c: Clima
# 1d: Qualidade dos dados

## 2. ECONOMICOS
# 2a: Mercado e Preços
# 2b: Produção e Aptidão
# 2c: Receitas
# 2d: Custos

## 3. INSTITUCIONAIS

##### 1.1 Criar df

In [59]:
df <- as_tibble(read.csv("base_modelos_h1_prec.csv")) 

df <- df %>%
  mutate(
    z0_codigo_ibge  = as.factor(z0_codigo_ibge),
    z0_ano  = as.integer(z0_ano),
    z0_municipio = as.factor(z0_municipio),
    z0_microrregiao = as.factor(z0_microrregiao),
    z0_bioma_predominante = as.factor(z0_bioma_predominante),
    x3_n_autos_flora__ibama_espacial = as.integer(x3_n_autos_flora__ibama_espacial),
    x3_n_autos_flora_elegiveis_lag1__ibama_espacial = as.integer(x3_n_autos_flora_elegiveis_lag1__ibama_espacial)
  )

##### 1.2 Visão Geral

In [60]:
head(df, 2)

z0_codigo_ibge,z0_ano,z0_municipio,z0_microrregiao,z0_bioma_predominante,z0_biome_amazonia_share_i,z0_biome_cerrado_share_i,z0_biome_pantanal_share_i,y1a_taxa_conversao__agricultura,y1a_conversao_ha__pastagem,y1a_taxa_conversao__pastagem,x2a_preco_real_usd2024_media_anual__soja,x2a_preco_real_usd2024_media_anual__milho,x2a_preco_real_usd2024_media_anual__carne_bovina,x2a_preco_real_usd2024_media_anual_lag1__soja,x2a_preco_real_usd2024_media_anual_lag1__milho,x2a_preco_real_usd2024_media_anual_lag1__carne_bovina,x2a_preco_real_usd2024_media_anual_lag2__soja,x2a_preco_real_usd2024_media_anual_lag2__milho,x2a_preco_real_usd2024_media_anual_lag2__carne_bovina,x2b_capacidade_valor__soja__gaez_yxHr0_soy,x2b_capacidade_valor__milho__gaez_yxHr0_mze,x2b_area_plantada_ha__milho,x2b_area_plantada_ha__soja,x2b_area_colhida_ha__milho,x2b_area_colhida_ha__soja,x2b_producao_t__milho,x2b_producao_t__soja,x2b_rendimento_medio_kg_ha__milho,x2b_rendimento_medio_kg_ha__soja,x2c_receita_bruta_estimada_usd2024_total__soja,x2c_receita_bruta_estimada_usd2024_total_lag1__soja,x2c_receita_bruta_estimada_usd2024_total_lag2__soja,x2c_receita_bruta_estimada_usd2024_total_media_lag1_lag2__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido_lag1__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido_lag2__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido_media_lag1_lag2__soja,x2c_receita_bruta_estimada_usd2024_total__milho,x2c_receita_bruta_estimada_usd2024_total_lag1__milho,x2c_receita_bruta_estimada_usd2024_total_lag2__milho,x2c_receita_bruta_estimada_usd2024_total_media_lag1_lag2__milho,x2c_receita_estimada__soja,x2c_lucro_precario__soja,x2c_receita_estimada__milho,x2c_lucro_precario__milho,x2c_receita_estimada__pecuaria,x2c_lucro_precario__pecuaria,x2d_custo_conab_soja_nao_declarada_safra1_conab_h_nominal,x2d_custo_conab_soja_nao_declarada_safra1_conab_h_usd2024,x2d_custo_imea_pecuaria_pecuaria_recria_engorda_safra0_coe_nominal,x2d_custo_imea_pecuaria_pecuaria_recria_engorda_safra0_cot_nominal,x2d_custo_imea_pecuaria_pecuaria_recria_engorda_safra0_ct_nominal,x2d_custo_conab_milho_2a_ogm_safra0_conab_h_nominal,x2d_custo_conab_milho_2a_ogm_safra0_conab_h_usd2024,x2d_custo_imea_agricola_soja_soja_gmo_safra1_imea_coe_nominal,x2d_custo_imea_agricola_soja_soja_gmo_safra1_imea_cot_nominal,x2d_custo_imea_agricola_soja_soja_gmo_safra1_imea_ct_nominal,x2d_custo_imea_pecuaria_pecuaria_ciclo_completo_safra0_coe_nominal,x2d_custo_imea_pecuaria_pecuaria_ciclo_completo_safra0_cot_nominal,x2d_custo_imea_pecuaria_pecuaria_ciclo_completo_safra0_ct_nominal,x2d_custo_conab_soja_ogm_safra1_conab_h_nominal,x2d_custo_conab_soja_ogm_safra1_conab_h_usd2024,x2d_custo_imea_agricola_milho_safra_nao_harmonizada_milho_alta_tecnologia_safra1_imea_coe_nominal,x2d_custo_imea_agricola_milho_safra_nao_harmonizada_milho_alta_tecnologia_safra1_imea_cot_nominal,x2d_custo_imea_agricola_milho_safra_nao_harmonizada_milho_alta_tecnologia_safra1_imea_ct_nominal,x2d_custo_conab_soja_convencional_explicita_safra1_conab_h_nominal,x2d_custo_conab_soja_convencional_explicita_safra1_conab_h_usd2024,x2d_custo_conab_milho_1a_nao_declarada_safra1_conab_h_nominal,x2d_custo_conab_milho_1a_nao_declarada_safra1_conab_h_usd2024,x2d_custo_conab_milho_2a_nao_declarada_safra0_conab_h_nominal,x2d_custo_conab_milho_2a_nao_declarada_safra0_conab_h_usd2024,x2d_custo_imea_pecuaria_pecuaria_cria_safra0_coe_nominal,x2d_custo_imea_pecuaria_pecuaria_cria_safra0_cot_nominal,x2d_custo_imea_pecuaria_pecuaria_cria_safra0_ct_nominal,x2d_custo_precario__soja,x2d_custo_precario__milho,x2d_custo_precario__pecuaria,x3_n_autos_flora__ibama_espacial,x3_n_autos_flora_elegiveis_lag1__ibama_espacial,y1a_conversao_ha__agricultura,x1b_y_nativa_inicial_ha,x3_flag_contagem_flora_parcial__ibama_espacial,x1c_v2__precip_mm_anual,x1c_v2__precip_anomaly_1981_2000,x1d_v2__meses_validos,x1d_v2__meses_cobertura_conhecida,x1d_v2__fracao_pixels_minima,x1d_v2__flag_ano_incompleto,x1c_v

In [61]:
tail(df, 2)

z0_codigo_ibge,z0_ano,z0_municipio,z0_microrregiao,z0_bioma_predominante,z0_biome_amazonia_share_i,z0_biome_cerrado_share_i,z0_biome_pantanal_share_i,y1a_taxa_conversao__agricultura,y1a_conversao_ha__pastagem,y1a_taxa_conversao__pastagem,x2a_preco_real_usd2024_media_anual__soja,x2a_preco_real_usd2024_media_anual__milho,x2a_preco_real_usd2024_media_anual__carne_bovina,x2a_preco_real_usd2024_media_anual_lag1__soja,x2a_preco_real_usd2024_media_anual_lag1__milho,x2a_preco_real_usd2024_media_anual_lag1__carne_bovina,x2a_preco_real_usd2024_media_anual_lag2__soja,x2a_preco_real_usd2024_media_anual_lag2__milho,x2a_preco_real_usd2024_media_anual_lag2__carne_bovina,x2b_capacidade_valor__soja__gaez_yxHr0_soy,x2b_capacidade_valor__milho__gaez_yxHr0_mze,x2b_area_plantada_ha__milho,x2b_area_plantada_ha__soja,x2b_area_colhida_ha__milho,x2b_area_colhida_ha__soja,x2b_producao_t__milho,x2b_producao_t__soja,x2b_rendimento_medio_kg_ha__milho,x2b_rendimento_medio_kg_ha__soja,x2c_receita_bruta_estimada_usd2024_total__soja,x2c_receita_bruta_estimada_usd2024_total_lag1__soja,x2c_receita_bruta_estimada_usd2024_total_lag2__soja,x2c_receita_bruta_estimada_usd2024_total_media_lag1_lag2__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido_lag1__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido_lag2__soja,x2c_receita_bruta_estimada_usd2024_por_ha_colhido_media_lag1_lag2__soja,x2c_receita_bruta_estimada_usd2024_total__milho,x2c_receita_bruta_estimada_usd2024_total_lag1__milho,x2c_receita_bruta_estimada_usd2024_total_lag2__milho,x2c_receita_bruta_estimada_usd2024_total_media_lag1_lag2__milho,x2c_receita_estimada__soja,x2c_lucro_precario__soja,x2c_receita_estimada__milho,x2c_lucro_precario__milho,x2c_receita_estimada__pecuaria,x2c_lucro_precario__pecuaria,x2d_custo_conab_soja_nao_declarada_safra1_conab_h_nominal,x2d_custo_conab_soja_nao_declarada_safra1_conab_h_usd2024,x2d_custo_imea_pecuaria_pecuaria_recria_engorda_safra0_coe_nominal,x2d_custo_imea_pecuaria_pecuaria_recria_engorda_safra0_cot_nominal,x2d_custo_imea_pecuaria_pecuaria_recria_engorda_safra0_ct_nominal,x2d_custo_conab_milho_2a_ogm_safra0_conab_h_nominal,x2d_custo_conab_milho_2a_ogm_safra0_conab_h_usd2024,x2d_custo_imea_agricola_soja_soja_gmo_safra1_imea_coe_nominal,x2d_custo_imea_agricola_soja_soja_gmo_safra1_imea_cot_nominal,x2d_custo_imea_agricola_soja_soja_gmo_safra1_imea_ct_nominal,x2d_custo_imea_pecuaria_pecuaria_ciclo_completo_safra0_coe_nominal,x2d_custo_imea_pecuaria_pecuaria_ciclo_completo_safra0_cot_nominal,x2d_custo_imea_pecuaria_pecuaria_ciclo_completo_safra0_ct_nominal,x2d_custo_conab_soja_ogm_safra1_conab_h_nominal,x2d_custo_conab_soja_ogm_safra1_conab_h_usd2024,x2d_custo_imea_agricola_milho_safra_nao_harmonizada_milho_alta_tecnologia_safra1_imea_coe_nominal,x2d_custo_imea_agricola_milho_safra_nao_harmonizada_milho_alta_tecnologia_safra1_imea_cot_nominal,x2d_custo_imea_agricola_milho_safra_nao_harmonizada_milho_alta_tecnologia_safra1_imea_ct_nominal,x2d_custo_conab_soja_convencional_explicita_safra1_conab_h_nominal,x2d_custo_conab_soja_convencional_explicita_safra1_conab_h_usd2024,x2d_custo_conab_milho_1a_nao_declarada_safra1_conab_h_nominal,x2d_custo_conab_milho_1a_nao_declarada_safra1_conab_h_usd2024,x2d_custo_conab_milho_2a_nao_declarada_safra0_conab_h_nominal,x2d_custo_conab_milho_2a_nao_declarada_safra0_conab_h_usd2024,x2d_custo_imea_pecuaria_pecuaria_cria_safra0_coe_nominal,x2d_custo_imea_pecuaria_pecuaria_cria_safra0_cot_nominal,x2d_custo_imea_pecuaria_pecuaria_cria_safra0_ct_nominal,x2d_custo_precario__soja,x2d_custo_precario__milho,x2d_custo_precario__pecuaria,x3_n_autos_flora__ibama_espacial,x3_n_autos_flora_elegiveis_lag1__ibama_espacial,y1a_conversao_ha__agricultura,x1b_y_nativa_inicial_ha,x3_flag_contagem_flora_parcial__ibama_espacial,x1c_v2__precip_mm_anual,x1c_v2__precip_anomaly_1981_2000,x1d_v2__meses_validos,x1d_v2__meses_cobertura_conhecida,x1d_v2__fracao_pixels_minima,x1d_v2__flag_ano_incompleto,x1c_v

In [62]:
dim(df)

[1] 3384  113

In [63]:
str(df)

tibble [3,384 × 113] (S3: tbl_df/tbl/data.frame)
 $ z0_codigo_ibge                                                                                   : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ z0_ano                                                                                           : int [1:3384] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ z0_municipio                                                                                     : Factor w/ 141 levels "Acorizal","Água Boa",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ z0_microrregiao                                                                                  : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ z0_bioma_predominante                                                                            : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ z0_biome_amazonia_share_i                                                                

In [64]:
kable(data.frame(Indice = 1:ncol(df), Coluna = names(df)))



| Indice|Coluna                                                                                            |
|------:|:-------------------------------------------------------------------------------------------------|
|      1|z0_codigo_ibge                                                                                    |
|      2|z0_ano                                                                                            |
|      3|z0_municipio                                                                                      |
|      4|z0_microrregiao                                                                                   |
|      5|z0_bioma_predominante                                                                             |
|      6|z0_biome_amazonia_share_i                                                                         |
|      7|z0_biome_cerrado_share_i                                                                          |
|      8|z0_biome

##### 1.3 Summary

In [65]:
## Pacote Base R
summary(df)

 z0_codigo_ibge     z0_ano             z0_municipio          z0_microrregiao
 5100102:  24   Min.   :2001   Acorizal      :  24   Norte Araguaia  : 336  
 5100201:  24   1st Qu.:2007   Água Boa      :  24   Jauru           : 288  
 5100250:  24   Median :2012   Alta Floresta :  24   Alto Teles Pires: 216  
 5100300:  24   Mean   :2012   Alto Araguaia :  24   Sinop           : 216  
 5100359:  24   3rd Qu.:2018   Alto Boa Vista:  24   Tesouro         : 216  
 5100409:  24   Max.   :2024   Alto Garças   :  24   Aripuanã        : 192  
 (Other):3240                  (Other)       :3240   (Other)         :1920  
 z0_bioma_predominante z0_biome_amazonia_share_i z0_biome_cerrado_share_i
 Amazônia:1824         Min.   :0.0000            Min.   :0.0000          
 Cerrado :1464         1st Qu.:0.0000            1st Qu.:0.0000          
 Pantanal:  96         Median :0.7111            Median :0.2526          
                       Mean   :0.5291            Mean   :0.4425          
              

In [66]:
# ## Summarytools
# dfSummary(df) %>%
#   print(method = "render", footnote = NA, style = "grid", graph.magnif = 0.75)

##### 1.4 Correlações

In [67]:
# options(repr.plot.width = 26, repr.plot.height = 26, repr.plot.res = 120)

# cor(df %>% select(where(is.numeric)), use = "pairwise.complete.obs") %>%
#   ggcorrplot(
#     hc.order = TRUE,
#     type = "lower",
#     method = "circle",
#     lab = TRUE,
#     lab_size = 1.8,
#     digits = 2,
#     tl.cex = 9,
#     tl.col = "grey20",
#     tl.srt = 90,
#     outline.color = "white",
#     colors = c("#D6604D", "white", "#4393C3"),
#     legend.title = "r de Pearson",
#     title = "Correlação entre variáveis numéricas",
#     ggtheme = theme_minimal
#   ) +
#   theme(
#     panel.grid = element_blank(),
#     axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5),
#     legend.position = "bottom",
#     plot.title = element_text(size = 18, face = "bold", hjust = 0.5),
#     plot.margin = margin(15, 15, 15, 15)
#   )


In [68]:
# options(repr.plot.width = 26, repr.plot.height = 26, repr.plot.res = 120)

# pairs(df[, c(9, 11:13)])

***

#### 2. MODELOS

In [69]:
## Ordem da análise
# Escolher variáveis
# Descrever o que se pretende com a análise
# Criar df
# Rodar modelo
# Interpretar resultados

In [70]:
## Efeitos fixos

# Testar:
#    - município
#    - microrregião
#    - bioma
#    - ano

##### 2.1 Modelos 1

In [71]:
# PRIMEIRA RODADA: CONVERSÃO PARA AGRICULTURA E INCENTIVO DA SOJA
# Pergunta: a associação entre preço da soja e conversão varia com a aptidão?
#
# y_agricultura: taxa de conversão para agricultura * 100 (porcentagem).
# ps1: log do preço da soja em t-1, centralizado nos 24 anos da base.
# aptidao_soja_pad: aptidão padronizada nos 141 municípios da base.
# xs1: ps1 * aptidao_soja_pad (interação preço x aptidão).
# rain: anomalia de precipitação / 100 (unidade: 100 mm).
#
# Amostra: flag de conversão observada para ambos os desfechos, como no roteiro.
# Referência para a futura regressão: efeitos fixos de município e ano.
# As demais colunas selecionadas são mantidas para as próximas variações.
# Esta etapa prepara o dataframe; ainda não estima o modelo.

###### Selecionar Variáveis

In [72]:
# 1. Selecionar as variáveis diretamente do df principal.
# Reexecute 1.1 Criar df para carregar o CSV ampliado antes desta seção.
df1 <- df %>%
  select(
    z0_codigo_ibge,
    z0_ano,
    z0_municipio,
    z0_microrregiao,
    z0_bioma_predominante,
    z0_biome_amazonia_share_i,
    z0_biome_cerrado_share_i,
    z0_biome_pantanal_share_i,

    y1a_taxa_conversao__agricultura,
    y1a_taxa_conversao__pastagem,

    x2a_preco_real_usd2024_media_anual__soja,
    x2a_preco_real_usd2024_media_anual_lag1__soja,
    x2b_capacidade_valor__soja__gaez_yxHr0_soy,
    x2a_preco_real_usd2024_media_anual__milho,
    x2a_preco_real_usd2024_media_anual_lag1__milho,
    x2b_capacidade_valor__milho__gaez_yxHr0_mze,
    x3_n_autos_flora__ibama_espacial,

    precip_anomalia = x1c_v2__precip_anomaly_1981_2000,
    amostra_observada = x4_v2__flag_amostra_conversao_ambos
  ) %>%
  mutate(
    amostra_observada = tolower(as.character(amostra_observada)) == "true"
  )

# Conferir as chaves e a flag antes de selecionar a amostra.
stopifnot(
  !anyNA(df1[c("z0_codigo_ibge", "z0_ano", "amostra_observada")]),
  !anyDuplicated(df1[c("z0_codigo_ibge", "z0_ano")]),
  nrow(df1) == nrow(df)
)

In [73]:
# 2. Calcular as referências na base completa, antes de filtrar a amostra.

# Separar um preço por ano. 
# a aptidão de cada município se repete ao longo dos anos. O resultado esperado é um vetor com 141 valores, um por município
preco_soja_anos <- df %>%
  distinct(z0_ano, x2a_preco_real_usd2024_media_anual_lag1__soja) %>% # mantém apenas as combinações diferentes de ano e preço.
  pull(x2a_preco_real_usd2024_media_anual_lag1__soja) # extrai a coluna de preços como um vetor numérico

# Separar uma aptidão por município
aptidao_soja_municipios <- df %>%
  distinct(z0_codigo_ibge, x2b_capacidade_valor__soja__gaez_yxHr0_soy) %>%
  pull(x2b_capacidade_valor__soja__gaez_yxHr0_soy)

# Conferir essas condições
stopifnot(
  length(preco_soja_anos) == n_distinct(df$z0_ano),
  length(aptidao_soja_municipios) == n_distinct(df$z0_codigo_ibge),
  all(is.finite(preco_soja_anos) & preco_soja_anos > 0),
  all(is.finite(aptidao_soja_municipios))
)

# Calcular as médias de referência
media_log_preco_soja <- mean(log(preco_soja_anos)) # calcula o logaritmo natural de cada preço e depois tira a média
media_aptidao_soja <- mean(aptidao_soja_municipios) # média da aptidão dos 141 municípios

# Desvio-padrão populacional, conforme o roteiro do Fefo (divisor N).
dp_aptidao_soja <- sqrt(mean(
  (aptidao_soja_municipios - media_aptidao_soja)^2
)) # Calcular a dispersão da aptidão
stopifnot(dp_aptidao_soja > 0)

###### Variáveis Derivadas

In [74]:
# 3. Criar as variáveis e selecionar a amostra observada.
# O preço com lag1 já corresponde a t-1: não aplicar outro lag.
df1 <- df1 %>%
  mutate(
    y_agricultura = 100 * y1a_taxa_conversao__agricultura,
    ps1 = log(x2a_preco_real_usd2024_media_anual_lag1__soja) -
      media_log_preco_soja,
    aptidao_soja_pad = (
      x2b_capacidade_valor__soja__gaez_yxHr0_soy - media_aptidao_soja
    ) / dp_aptidao_soja,
    xs1 = ps1 * aptidao_soja_pad,
    rain = precip_anomalia / 100
  ) %>%
  filter(amostra_observada)

# Não excluir silenciosamente linhas com dados ausentes nas variáveis do modelo.
# Verificar: Ausência de valores faltantes nas chaves e na flag, Uma única observação por município e ano e Mesma quantidade de linhas da base principal
stopifnot(all(is.finite(as.matrix(
  select(df1, y_agricultura, ps1, aptidao_soja_pad, xs1, rain)
))))

###### Visão Geral

In [75]:
head(df1, 2)

z0_codigo_ibge,z0_ano,z0_municipio,z0_microrregiao,z0_bioma_predominante,z0_biome_amazonia_share_i,z0_biome_cerrado_share_i,z0_biome_pantanal_share_i,y1a_taxa_conversao__agricultura,y1a_taxa_conversao__pastagem,x2a_preco_real_usd2024_media_anual__soja,x2a_preco_real_usd2024_media_anual_lag1__soja,x2b_capacidade_valor__soja__gaez_yxHr0_soy,x2a_preco_real_usd2024_media_anual__milho,x2a_preco_real_usd2024_media_anual_lag1__milho,x2b_capacidade_valor__milho__gaez_yxHr0_mze,x3_n_autos_flora__ibama_espacial,precip_anomalia,amostra_observada,y_agricultura,ps1,aptidao_soja_pad,xs1,rain
<fct>,<int>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5100102,2001,Acorizal,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.01732317,346.9711,385.9825,3665.637,158.8466,161.3827,4604.177,0,82.17429,TRUE,0.008563177,-0.3467312,1.960358,-0.6797172,0.8217429
5100102,2002,Acorizal,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.01178126,370.6660,346.9711,3665.637,173.0334,158.8466,4604.177,0,-150.84732,TRUE,0.002493791,-0.4532818,1.960358,-0.8885945,-1.5084732


In [76]:
# 4. Conferir a amostra e as variáveis que usaremos no primeiro modelo.
df1 %>%
  summarise(
    observacoes = n(),
    municipios = n_distinct(z0_codigo_ibge),
    anos = n_distinct(z0_ano),
    primeiro_ano = min(z0_ano),
    ultimo_ano = max(z0_ano)
  )

observacoes,municipios,anos,primeiro_ano,ultimo_ano
<int>,<int>,<int>,<int>,<int>
2772,137,24,2001,2024


In [77]:
str(df1)

tibble [2,772 × 24] (S3: tbl_df/tbl/data.frame)
 $ z0_codigo_ibge                                : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ z0_ano                                        : int [1:2772] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ z0_municipio                                  : Factor w/ 141 levels "Acorizal","Água Boa",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ z0_microrregiao                               : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ z0_bioma_predominante                         : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ z0_biome_amazonia_share_i                     : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $ z0_biome_cerrado_share_i                      : num [1:2772] 1 1 1 1 1 1 1 1 1 1 ...
 $ z0_biome_pantanal_share_i                     : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $ y1a_taxa_conversao__agricultura               : num [1:2772] 8.56e-05 2.49e-05 2.

###### model_1a

In [84]:
# model_1a: df1, efeitos fixos de município e ano.
# Erros-padrão agrupados por município, conforme o roteiro.
model_1a <- fixest::feols(
  y_agricultura ~ xs1 + rain | 
    z0_codigo_ibge + z0_ano, # efeitos fixos de município e ano
    data = df1, # dados do modelo
    vcov = ~z0_codigo_ibge # erros-padrão agrupados por município
)

NOTE: 3/0 fixed-effect singletons were removed (3 observations).



In [85]:
summary(model_1a)

OLS estimation, Dep. Var.: y_agricultura
Observations: 2,769
Fixed-effects: z0_codigo_ibge: 134,  z0_ano: 24
Standard-errors: Clustered (z0_codigo_ibge) 
      Estimate Std. Error  t value Pr(>|t|)    
xs1   0.136338   0.053204  2.56256 0.011504 *  
rain -0.013851   0.006709 -2.06449 0.040914 *  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
RMSE: 0.298978     Adj. R2: 0.326561
                 Within R2: 0.017409